In [0]:
%run ../configs/config

In [0]:
cards = f'{bronze_schema}.cards'
cards_df = spark.table(cards)

In [0]:
cards_df.printSchema()

In [0]:
cards_dropped = cards_df.dropDuplicates().filter(F.col('card_id').isNotNull())


In [0]:
cards_renamed = (
        cards_dropped
            .withColumnsRenamed({'type':'card_type', 'issued': 'date_issued'})
)

In [0]:

cards_format = (cards_renamed
                .withColumn('date_issued',   F.to_date(F.col('date_issued'), 'yyMMdd HH:mm:ss').cast('datedisplay'))
                .withColumn('card_type', F.initcap(F.col('card_type')))


)


In [0]:
cards_final_df = add_metadata_columns(cards_format)

In [0]:
display(cards_final_df)

In [0]:
(
    cards_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{silver_schema}.cards')
)

In [0]:
display(spark.table(f'{silver_schema}.cards'))